# 05 XAI — Análise Explicável com LightGBM

**Repositório:** [FABRICIOBARILI/DOUTORADO](https://github.com/FABRICIOBARILI/DOUTORADO)
Dados: Google Cloud Storage | Projeto GCP: `doutorado-501917` | Bucket: `2025_rides`

> **DADOS MOVIDOS DO GOOGLE DRIVE PARA O GOOGLE CLOUD STORAGE.** Ganho em velocidade de leitura.

In [ ]:
# ── SINCRONIZAR COM GITHUB ──────────────────────────────────────────────────
# Execute esta célula para puxar a versão mais recente do repositório.
import os

REPO_URL = "https://github.com/FABRICIOBARILI/DOUTORADO.git"
REPO_DIR = "/content/DOUTORADO"
BRANCH   = "feat/changelog-inicial"   # ajuste conforme o branch ativo

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Diretório atual: {os.getcwd()}")

In [ ]:
!pip install -q gcsfs duckdb
import pandas as pd
import gcsfs
from google.colab import auth
import duckdb

# 1. Garante a autenticação nativa do Colab
auth.authenticate_user()

# 2. Inicializa o FileSystem do GCS apontando para o seu projeto
project_id = 'doutorado-501917'
bucket_name = '2025_rides'
fs = gcsfs.GCSFileSystem(project=project_id)

# 3. Integração Mágica: Registra o gcsfs no DuckDB
# Isso faz o DuckDB usar a autenticação do Colab automaticamente
duckdb.register_filesystem(fs)

# 4. Usa o glob do gcsfs para encontrar todos os arquivos parquet na pasta
file_pattern = f"gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet"
print(f"Buscando arquivos com o padrão: {file_pattern}...")
file_list = fs.glob(file_pattern)
print(f"Encontrados {len(file_list)} arquivos Parquet. Preparando leitura...")

# Adiciona o prefixo gs:// para o DuckDB reconhecer corretamente usando o fsspec/gcsfs
gs_file_list = [f"gs://{f}" for f in file_list]



Buscando arquivos com o padrão: gs://2025_rides/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet...
Encontrados 3531 arquivos Parquet. Preparando leitura...


In [ ]:

# 5. Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

print("Carregando os dados com DuckDB...")

# 6. Cria a query passando a lista de arquivos e executa retornando para DataFrame pandas
# Limitando a saída de string para evitar erros de formatação na query
files_sql_array = ", ".join([f"'{f}'" for f in gs_file_list])

query = f"""
    SELECT *
    FROM read_parquet([{files_sql_array}])
"""

#df = duckdb.sql(query).df()

#print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files using DuckDB.")
#display(df.head())

Carregando os dados com DuckDB...


In [ ]:
# 1. Autentique sua conta do Google Cloud
#from google.colab import auth
#auth.authenticate_user()

# 2. Defina o ID do seu projeto no GCP e o nome do seu Bucket
#project_id = 'DOUTORADO'
#bucket_name = '2025_rides'

#!gcloud config set project {project_id}

# 3. Monte o seu Google Drive no Colab
#from google.colab import drive
#drive.mount('/content/drive')

#caminhos_parquet = [
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_4/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_5/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_6/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_7/trips_log/_staging/**/*.parquet',
#]

# 4. Copie os dados do Drive direto para o GCS usando o comando otimizado gsutil
# O parâmetro '-m' ativa a cópia em paralelo de múltiplos arquivos
#!gsutil -m cp -r /content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_* gs://{bucket_name}/


# INÍCIO DA LEITURA, TRATAMENTO E ANÁLISE DOS DADOS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
datasets_dir = "/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS"
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/dados_meteorologicos_utci_horario.csv", "./")
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/DADOS_AEROPORTO/03_voos_atrasados_sbpa.csv", "./")
shutil.copy(f"{datasets_dir}/Aeroporto_Salgado_Filho_h3_res12.csv", "./")

import pandas as pd

# Carregando os datasets
df_clima = pd.read_csv('/content/dados_meteorologicos_utci_horario.csv')
df_voos = pd.read_csv('/content/03_voos_atrasados_sbpa.csv', sep=";")
df_h3 = pd.read_csv('/content/Aeroporto_Salgado_Filho_h3_res12.csv')

# Exibindo as primeiras linhas de cada um para verificar
print("--- Dados Meteorológicos ---")
#display(df_clima.head())

print("\n--- Voos Atrasados ---")
display(df_voos.head())

print("\n--- Aeroporto Salgado Filho (H3) ---")
#display(df_h3.head())

--- Dados Meteorológicos ---

--- Voos Atrasados ---


,ICAO_EMPRESA_AEREA,NUMERO_VOO,CODIGO_AUTORIZACAO_DI,CODIGO_TIPO_LINHA,ICAO_AERODROMO_ORIGEM,ICAO_AERODROMO_DESTINO,PARTIDA_PREVISTA,PARTIDA_REAL,CHEGADA_PREVISTA,CHEGADA_REAL,SITUACAO_VOO,CODIGO_JUSTIFICATIVA,ATRASO_MINUTOS,STATUS_ATRASO,DATA_PREVISTA,DIA_SEMANA,NOME_DIA,HORA_PREVISTA
0,GLO,1885,0,N,SBGR,SBPA,2025-01-23 14:50:00,2025-01-23 15:09:00,2025-01-23 16:35:00,2025-01-23 16:57:00,REALIZADO,NaN,22.0,Atraso Leve (15-59 min),2025-01-23,3,Quinta,16
1,GLO,1885,0,N,SBGR,SBPA,2025-01-26 14:50:00,2025-01-26 15:12:00,2025-01-26 16:35:00,2025-01-26 16:59:00,REALIZADO,NaN,24.0,Atraso Leve (15-59 min),2025-01-26,6,Domingo,16
2,GLO,1885,0,N,SBGR,SBPA,2025-01-29 14:50:00,2025-01-29 14:50:00,2025-01-29 16:35:00,2025-01-29 17:07:00,REALIZADO,NaN,32.0,Atraso Leve (15-59 min),2025-01-29,2,Quarta,16
3,LPE,2422,0,I,SPJC,SBPA,2025-01-06 01:50:00,2025-01-06 02:38:00,2025-01-06 06:30:00,2025-01-06 07:05:00,REALIZADO,NaN,35.0,Atraso Leve (15-59 min),2025-01-06,0,Segunda,6
4,LPE,2422,0,I,SPJC,SBPA,2025-01-20 01:50:00,2025-01-20 02:33:00,2025-01-20 06:30:00,2025-01-20 07:01:00,REALIZADO,NaN,31.0,Atraso Leve (15-59 min),2025-01-20,0,Segunda,6



--- Aeroporto Salgado Filho (H3) ---


In [ ]:
print("--- Tipos de dados: Dados Meteorológicos (df_clima) ---")
#display(df_clima.dtypes)

print("\n--- Tipos de dados: Voos Atrasados (df_voos) ---")
#display(df_voos.dtypes)

print("\n--- Tipos de dados: Aeroporto Salgado Filho H3 (df_h3) ---")
#display(df_h3.dtypes)


--- Tipos de dados: Dados Meteorológicos (df_clima) ---

--- Tipos de dados: Voos Atrasados (df_voos) ---

--- Tipos de dados: Aeroporto Salgado Filho H3 (df_h3) ---


In [ ]:
import gcsfs
import duckdb

# Garante que o filesystem gcsfs esteja registrado no DuckDB
fs = gcsfs.GCSFileSystem(project=project_id)
try:
    duckdb.register_filesystem(fs)
except Exception:
    pass # Ignora caso já esteja registrado

# Caminhos base das simulações
caminhos_base = [
    f'gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_4/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_5/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_6/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_7/trips_log/_staging/**/*.parquet',
]

print("Buscando todos os arquivos Parquet nos diretórios (isso pode levar alguns instantes)...")
todos_arquivos = []
for caminho in caminhos_base:
    arquivos = fs.glob(caminho)
    todos_arquivos.extend([f"gs://{f}" for f in arquivos])

print(f"Total de {len(todos_arquivos):,} arquivos Parquet encontrados.")

# Criando o array de strings para injetar na query do DuckDB
files_sql_array = ", ".join([f"'{f}'" for f in todos_arquivos])


Buscando todos os arquivos Parquet nos diretórios (isso pode levar alguns instantes)...
Total de 12,368 arquivos Parquet encontrados.


In [ ]:
print("Preparando a query principal para executar com DuckDB...")
print("Lendo do GCS com a nova integração e aplicando undersampling...")

# Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

# A query balanceada
inicio_ts = 1735699200
fim_ts = 1767235200

# Combine all conditions into a single query with appropriate sampling
# Substituímos {caminhos_parquet} por [{files_sql_array}]
query_combined = f"""
    SELECT
        request_ts,
        event_name,
        origin_h3,
        CASE
            WHEN UPPER(event_name) LIKE '%ATRASADO%' THEN 'DS_VOO'
            WHEN UPPER(event_name) LIKE '%SEVERIDADE%' THEN 'DS_CLIMA'
            ELSE 'DS_OUTROS'
        END AS dataset_type
    FROM read_parquet([{files_sql_array}], hive_partitioning = true)
    WHERE request_ts >= {inicio_ts}
      AND request_ts < {fim_ts}
    USING SAMPLE 30 PERCENT
"""


Preparando a query principal para executar com DuckDB...
Lendo do GCS com a nova integração e aplicando undersampling...


In [ ]:
# Instala a biblioteca necessária
!pip install google-cloud-storage

from google.cloud import storage
from google.colab import auth

# Autenticação
auth.authenticate_user()

# Crie um cliente apontando para o seu projeto
project_id = 'doutorado-501917'
client = storage.Client(project=project_id)

# Acesse o bucket e o arquivo específico
bucket_name = '2025_rides'
bucket = client.get_bucket(bucket_name)

In [ ]:
#!pip install -q gcsfs
#import pandas as pd
#import gcsfs
#from google.colab import auth

# Garante a autenticação nativa do Colab
#auth.authenticate_user()

# Inicializa o FileSystem do GCS apontando para o seu projeto
#fs = gcsfs.GCSFileSystem(project='doutorado-501917')

# Usa o glob do gcsfs para encontrar todos os arquivos parquet na pasta
#file_pattern = f"gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet"
#print(f"Buscando arquivos com o padrão: {file_pattern}...")

#file_list = fs.glob(file_pattern)
#print(f"Encontrados {len(file_list)} arquivos Parquet. Carregando os dados...")

# Adiciona o prefixo gs:// de volta para o pandas reconhecer e lê a lista
# Passamos o filesystem para evitar problemas de autenticação internos
#df = pd.read_parquet([f"gs://{f}" for f in file_list], filesystem=fs)

#print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files.")
#display(df.head())


In [ ]:
# Executa a query combinada e converte diretamente para DataFrame Pandas

# --- BEGIN FIX: Configure DuckDB for Google Cloud Storage (GCS) ---
# Install and load the httpfs extension for remote file system access
#duckdb.sql("INSTALL httpfs;")
#duckdb.sql("LOAD httpfs;")

# Import necessary libraries for Google Cloud authentication
#import google.auth
#import google.auth.transport.requests

# Get default credentials and refresh them to obtain an access token
#credentials, project = google.auth.default()
#auth_req = google.auth.transport.requests.Request()
#credentials.refresh(auth_req)
#access_token = credentials.token


# 5. Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

print("Carregando os dados com DuckDB...")

# 6. Cria a query passando a lista de arquivos e executa retornando para DataFrame pandas
# Limitando a saída de string para evitar erros de formatação na query
files_sql_array = ", ".join([f"'{f}'" for f in gs_file_list])

df = duckdb.sql(query_combined).df()

print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files using DuckDB.")
display(df.head())

df_combined = duckdb.sql(query_combined).df()

# Filtra para criar os datasets DS_VOO, DS_CLIMA e DS_OUTROS
DS_VOO = df_combined[df_combined['dataset_type'] == 'DS_VOO'].drop(columns=['dataset_type'])
DS_CLIMA = df_combined[df_combined['dataset_type'] == 'DS_CLIMA'].drop(columns=['dataset_type'])
DS_OUTROS = df_combined[df_combined['dataset_type'] == 'DS_OUTROS'].drop(columns=['dataset_type'])

print(f"🚀 DATASET DE ATRASOS (DS_VOO): {len(DS_VOO):,}")
print(f"🚀 DATASET DE ATRASOS (DS_CLIMA): {len(DS_CLIMA):,}")
print(f"🚀 DATASET DE ATRASOS (DS_OUTROS): {len(DS_OUTROS):,}")
display(DS_VOO.head())

Carregando os dados com DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Successfully read 60,035,671 rows from GCS parquet files using DuckDB.


,request_ts,event_name,origin_h3,dataset_type
0,1735714951,,8ca90139292b1ff,DS_OUTROS
1,1735716014,,8ca9012a48557ff,DS_OUTROS
2,1735718501,,8ca90e9ad0cc3ff,DS_OUTROS
3,1735721351,,8ca90176db4a7ff,DS_OUTROS
4,1735722526,,8ca90128391d3ff,DS_OUTROS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🚀 DATASET DE ATRASOS (DS_VOO): 199,858
🚀 DATASET DE ATRASOS (DS_CLIMA): 317,707
🚀 DATASET DE ATRASOS (DS_OUTROS): 59,794,468


,request_ts,event_name,origin_h3
3960,1735898854,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e934104bff
4622,1735901421,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9340b59ff
5938,1735933991,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e935c9a3ff
6236,1735898747,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e935ca81ff
6253,1735900494,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e930d26dff


In [ ]:
import pandas as pd
import numpy as np

print("Iniciando a normalização do tamanho dos datasets DS_VOO, DS_CLIMA e DS_OUTROS...")

# Definir o período de interesse para a distribuição (2025 inteiro)
start_date = pd.to_datetime('2025-01-01 00:00:00')
end_date = pd.to_datetime('2025-12-31 23:59:59')

# Convert UNIX timestamps in seconds to datetime objects
DS_VOO['request_ts_dt'] = pd.to_datetime(DS_VOO['request_ts'], unit='s')
DS_CLIMA['request_ts_dt'] = pd.to_datetime(DS_CLIMA['request_ts'], unit='s')
DS_OUTROS['request_ts_dt'] = pd.to_datetime(DS_OUTROS['request_ts'], unit='s')

print("Corrected datetime format:")
display(DS_VOO[['request_ts', 'request_ts_dt']].head())

Iniciando a normalização do tamanho dos datasets DS_VOO, DS_CLIMA e DS_OUTROS...
Corrected datetime format:


,request_ts,request_ts_dt
3960,1735898854,2025-01-03 10:07:34
4622,1735901421,2025-01-03 10:50:21
5938,1735933991,2025-01-03 19:53:11
6236,1735898747,2025-01-03 10:05:47
6253,1735900494,2025-01-03 10:34:54


In [ ]:
# Filtrar cada DataFrame para o período de 2025
print(f"Filtrando datasets para o período de {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}...")
DS_VOO_2025 = DS_VOO[(DS_VOO['request_ts_dt'] >= start_date) & (DS_VOO['request_ts_dt'] <= end_date)]
DS_CLIMA_2025 = DS_CLIMA[(DS_CLIMA['request_ts_dt'] >= start_date) & (DS_CLIMA['request_ts_dt'] <= end_date)]
DS_OUTROS_2025 = DS_OUTROS[(DS_OUTROS['request_ts_dt'] >= start_date) & (DS_OUTROS['request_ts_dt'] <= end_date)]

print(f"Tamanho de DS_VOO_2025 após filtro: {len(DS_VOO_2025):,} linhas")
print(f"Tamanho de DS_CLIMA_2025 após filtro: {len(DS_CLIMA_2025):,} linhas")
print(f"Tamanho de DS_OUTROS_2025 após filtro: {len(DS_OUTROS_2025):,} linhas")

# Encontrar o menor tamanho entre os DataFrames filtrados
min_size = min(len(DS_VOO_2025), len(DS_CLIMA_2025), len(DS_OUTROS_2025))
print(f"\nO menor tamanho entre os datasets filtrados é: {min_size:,} linhas.")

# Realizar o subsampling (amostragem aleatória) para equalizar o tamanho, mantendo a distribuição temporal
print(f"Subsampling todos os datasets para {min_size:,} linhas...")

if len(DS_VOO_2025) > min_size:
    DS_VOO_NORMALIZED = DS_VOO_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_VOO_NORMALIZED = DS_VOO_2025.sort_values('request_ts_dt').reset_index(drop=True)

if len(DS_CLIMA_2025) > min_size:
    DS_CLIMA_NORMALIZED = DS_CLIMA_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_CLIMA_NORMALIZED = DS_CLIMA_2025.sort_values('request_ts_dt').reset_index(drop=True)

if len(DS_OUTROS_2025) > min_size:
    DS_OUTROS_NORMALIZED = DS_OUTROS_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_OUTROS_NORMALIZED = DS_OUTROS_2025.sort_values('request_ts_dt').reset_index(drop=True)

# Atualizar os DataFrames originais com os resultados normalizados
DS_VOO = DS_VOO_NORMALIZED
DS_CLIMA = DS_CLIMA_NORMALIZED
DS_OUTROS = DS_OUTROS_NORMALIZED

print("\n✅ Normalização concluída!")
print(f"Novo tamanho de DS_VOO: {len(DS_VOO):,} linhas")
print(f"Novo tamanho de DS_CLIMA: {len(DS_CLIMA):,} linhas")
print(f"Novo tamanho de DS_OUTROS: {len(DS_OUTROS):,} linhas")

print("\nVerificação das primeiras linhas de cada dataset normalizado:")
print("\nDS_VOO (Normalizado):")
display(DS_VOO.head())

print("\nDS_CLIMA (Normalizado):")
display(DS_CLIMA.head())

print("\nDS_OUTROS (Normalizado):")
display(DS_OUTROS.head())

Filtrando datasets para o período de 2025-01-01 a 2025-12-31...
Tamanho de DS_VOO_2025 após filtro: 199,858 linhas
Tamanho de DS_CLIMA_2025 após filtro: 317,707 linhas
Tamanho de DS_OUTROS_2025 após filtro: 59,794,468 linhas

O menor tamanho entre os datasets filtrados é: 199,858 linhas.
Subsampling todos os datasets para 199,858 linhas...

✅ Normalização concluída!
Novo tamanho de DS_VOO: 199,858 linhas
Novo tamanho de DS_CLIMA: 199,858 linhas
Novo tamanho de DS_OUTROS: 199,858 linhas

Verificação das primeiras linhas de cada dataset normalizado:

DS_VOO (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735898423,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9358d99ff,2025-01-03 10:00:23
1,1735898430,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9341731ff,2025-01-03 10:00:30
2,1735898470,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e934b651ff,2025-01-03 10:01:10
3,1735898471,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90129b4dadff,2025-01-03 10:01:11
4,1735898477,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9358c3dff,2025-01-03 10:01:17



DS_CLIMA (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735718408,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935ce53ff,2025-01-01 08:00:08
1,1735718413,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9342415ff,2025-01-01 08:00:13
2,1735718437,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935c065ff,2025-01-01 08:00:37
3,1735718452,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358ccdff,2025-01-01 08:00:52
4,1735718515,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358e51ff,2025-01-01 08:01:55



DS_OUTROS (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735704844,,8ca901298a137ff,2025-01-01 04:14:04
1,1735705452,,8ca90e91b235bff,2025-01-01 04:24:12
2,1735705753,,8ca90e8021765ff,2025-01-01 04:29:13
3,1735705838,,8ca90e835946bff,2025-01-01 04:30:38
4,1735705994,,8ca90e9030147ff,2025-01-01 04:33:14


In [ ]:
print(f"🚀 DATASET DE EVENTOS CLIMÁTICOS (DS_CLIMA): {len(DS_CLIMA):,}")
display(DS_CLIMA.head())

🚀 DATASET DE EVENTOS CLIMÁTICOS (DS_CLIMA): 199,858


,request_ts,event_name,origin_h3,request_ts_dt
0,1735718408,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935ce53ff,2025-01-01 08:00:08
1,1735718413,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9342415ff,2025-01-01 08:00:13
2,1735718437,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935c065ff,2025-01-01 08:00:37
3,1735718452,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358ccdff,2025-01-01 08:00:52
4,1735718515,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358e51ff,2025-01-01 08:01:55


In [ ]:
# Executa a query e converte diretamente para DataFrame Pandas
print(f"🚀 DATASET DE OUTROS EVENTOS (DS_OUTROS): {len(DS_OUTROS):,}")
display(DS_OUTROS.head())

🚀 DATASET DE OUTROS EVENTOS (DS_OUTROS): 199,858


,request_ts,event_name,origin_h3,request_ts_dt
0,1735704844,,8ca901298a137ff,2025-01-01 04:14:04
1,1735705452,,8ca90e91b235bff,2025-01-01 04:24:12
2,1735705753,,8ca90e8021765ff,2025-01-01 04:29:13
3,1735705838,,8ca90e835946bff,2025-01-01 04:30:38
4,1735705994,,8ca90e9030147ff,2025-01-01 04:33:14


In [ ]:
len_atraso = len(DS_VOO)
len_climatico = len(DS_CLIMA)
len_nao_evento = len(DS_OUTROS)

total = len_atraso + len_climatico + len_nao_evento

print(f"Total de linhas: {total:,}\n")
print(f"Atraso de Voo (DS_VOO): {len_atraso:,} ({len_atraso/total:.2%})")
print(f"Evento Climático (DS_CLIMA): {len_climatico:,} ({len_climatico/total:.2%})")
print(f"Outros Eventos (DS_OUTROS): {len_nao_evento:,} ({len_nao_evento/total:.2%})")

Total de linhas: 599,574

Atraso de Voo (DS_VOO): 199,858 (33.33%)
Evento Climático (DS_CLIMA): 199,858 (33.33%)
Outros Eventos (DS_OUTROS): 199,858 (33.33%)


In [ ]:
# Convertendo request_ts para datetime nos 3 datasets
#DS_VOO['request_ts_dt'] = pd.to_datetime(DS_VOO['request_ts'], unit='s')
#DS_CLIMA['request_ts_dt'] = pd.to_datetime(DS_CLIMA['request_ts'], unit='s')
#DS_OUTROS['request_ts_dt'] = pd.to_datetime(DS_OUTROS['request_ts'], unit='s')

# Exibindo uma amostra rápida de cada um para verificação
print("Atrasos de Voo (DS_VOO):")
display(DS_VOO[['request_ts', 'request_ts_dt']].head(2))

print("\nEventos Climáticos (DS_CLIMA):")
display(DS_CLIMA[['request_ts', 'request_ts_dt']].head(2))

print("\nOutros Eventos (DS_OUTROS):")
display(DS_OUTROS[['request_ts', 'request_ts_dt']].head(2))

Atrasos de Voo (DS_VOO):


,request_ts,request_ts_dt
0,1735898423,2025-01-03 10:00:23
1,1735898430,2025-01-03 10:00:30



Eventos Climáticos (DS_CLIMA):


,request_ts,request_ts_dt
0,1735718408,2025-01-01 08:00:08
1,1735718413,2025-01-01 08:00:13



Outros Eventos (DS_OUTROS):


,request_ts,request_ts_dt
0,1735704844,2025-01-01 04:14:04
1,1735705452,2025-01-01 04:24:12


### 🚀 Estratégia para Criação do Algoritmo Preditivo

Com base nas análises feitas (Clima + Voos), a melhor estratégia para sair de modelos *explicativos* para um modelo **preditivo** robusto envolve 5 pilares fundamentais:

#### 1. Criação da "Base Master" (Unificação)
Até agora, avaliamos o clima e os voos separadamente. O modelo preditivo precisará de tudo na mesma linha do tempo.
*   **Ação:** Fazer um merge combinando o `df_ts` (Eventos) com o `df_clima` (variáveis meteorológicas topo) **E** agregando informações do `df_voos` (ex: número de voos previstos para pousar naquela janela de 2 horas, mix de empresas aéreas, etc.).

#### 2. Engenharia de Features (Feature Engineering)
Além dos dados brutos, precisamos dar contexto ao algoritmo.
*   **Temporais:** Extrair `Hora do Dia`, `Mês`, `Dia da Semana`, e `Trimestre` (como vimos, o Q3 é crítico).
*   **Lags (Defasagens):** Usar o clima das horas *anteriores* para prever a próxima hora (ex: se a pressão atmosférica está caindo nas últimas 3 horas, a chance de evento aumenta).

#### 3. Tratamento de Desbalanceamento Severo
O seu *target* (eventos) representa menos de 1% da base total (ex: 38k contra 4.4M). Se não tratarmos, o modelo vai sempre prever "0" e acertar 99% das vezes, mas falhar no que importa.
*   **Ação:** Usar parâmetros nativos como `is_unbalance=True` ou `scale_pos_weight` no LightGBM/XGBoost.
*   **Alternativa:** Técnicas de reamostragem como *Undersampling* da classe majoritária no treino (você já fez um sampling para evitar OOM, podemos otimizar isso).

#### 4. Validação Temporal (Time-Series Split)
**Nunca** use um `train_test_split` aleatório em dados que dependem do tempo, pois isso gera *Data Leakage* (vazar o futuro para prever o passado).
*   **Ação:** Separar os dados cronologicamente. Treinar com os primeiros 9 meses de 2025 (Jan-Set) e testar com os 3 meses seguintes (Out-Dez) para simular o uso no mundo real.

#### 5. Métricas de Avaliação Corretas
*Acurácia* não serve aqui.
*   Focar no **Recall** (capacidade de detectar todos os eventos reais, minimizando falsos negativos) e **Precision** (quando dá o alerta, qual a chance de ser real).
*   Analisar a curva **PR-AUC** (Precision-Recall Area Under Curve), que é a métrica padrão-ouro para dados altamente desbalanceados.

---
**Próximo Passo Prático:** gerar código para construir a **Base Master** unindo Clima + Voos + features temporais?

In [ ]:
import pandas as pd
import numpy as np

print("1. Padronizando as colunas temporais e criando janelas de 4 horas...")
# Garantir que as colunas de tempo sejam datetime
df_clima['time'] = pd.to_datetime(df_clima['time'])
df_voos['CHEGADA_REAL'] = pd.to_datetime(df_voos['CHEGADA_REAL'], errors='coerce')

# Criar as janelas de 4 horas
df_clima['time_window'] = df_clima['time'].dt.floor('4h')
df_voos['time_window'] = df_voos['CHEGADA_REAL'].dt.floor('4h')

# Garantir datetime e criar janelas para os eventos
dataframes_eventos = [DS_OUTROS, DS_CLIMA, DS_VOO]
for df in dataframes_eventos:
    df['request_ts_dt'] = pd.to_datetime(df['request_ts_dt'])
    df['time_window'] = df['request_ts_dt'].dt.floor('4h')

print("2. Definindo o TARGET numérico para cada classe...")
# 0: Não Evento | 1: Evento Climático | 2: Atraso de Voo
DS_OUTROS['target'] = 0
DS_CLIMA['target'] = 1
DS_VOO['target'] = 2

# Concatenar todos os eventos na base principal
df_eventos_combinados = pd.concat([
    DS_OUTROS[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_CLIMA[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_VOO[['time_window', 'origin_h3', 'target', 'request_ts_dt']]
], ignore_index=True)

print("3. Agregando Clima e Voos nas janelas...")
# Clima: tira a média das variáveis meteorológicas nas janelas
df_clima_agg = df_clima.drop(columns=['time']).groupby('time_window').mean(numeric_only=True).reset_index()

# Voos: contabiliza e preserva as informações detalhadas em listas para cada janela
df_voos_agg = df_voos.dropna(subset=['time_window']).groupby('time_window').agg(
    qtd_voos_previstos=('NUMERO_VOO', 'count'),
    qtd_empresas_aereas=('ICAO_EMPRESA_AEREA', lambda x: x.nunique()),
    lista_chegada_real=('CHEGADA_REAL', lambda x: list(x)),
    lista_empresas_aereas=('ICAO_EMPRESA_AEREA', lambda x: list(x)),
    lista_numeros_voo=('NUMERO_VOO', lambda x: list(x)),
    lista_codigo_linha=('CODIGO_TIPO_LINHA', lambda x: list(x))
).reset_index()

print("4. Montando a Base Master (Merges)...")
# Unindo Eventos + Clima Agregado
df_master = pd.merge(df_eventos_combinados, df_clima_agg, on='time_window', how='left')

# Unindo com os Voos Agregados
df_master = pd.merge(df_master, df_voos_agg, on='time_window', how='left')

# Preenchendo nulos onde não havia voos na janela com 0
df_master['qtd_voos_previstos'] = df_master['qtd_voos_previstos'].fillna(0)
df_master['qtd_empresas_aereas'] = df_master['qtd_empresas_aereas'].fillna(0)

# Ordenar a base cronologicamente
df_master = df_master.sort_values('time_window').reset_index(drop=True)

print("\n✅ Base Master Finalizada e pronta para ML!")
print(f"Tamanho final da base: {len(df_master):,} linhas e {df_master.shape[1]} colunas")
print("\nDistribuição do Target:")
target_map = {0: '0 (Não Evento)', 1: '1 (Evento Climático)', 2: '2 (Atraso de Voo)'}
print(df_master['target'].map(target_map).value_counts())

display(df_master.head())


1. Padronizando as colunas temporais e criando janelas de 4 horas...
2. Definindo o TARGET numérico para cada classe...
3. Agregando Clima e Voos nas janelas...
4. Montando a Base Master (Merges)...

✅ Base Master Finalizada e pronta para ML!
Tamanho final da base: 599,574 linhas e 54 colunas

Distribuição do Target:
target
0 (Não Evento)          199858
1 (Evento Climático)    199858
2 (Atraso de Voo)       199858
Name: count, dtype: int64


,time_window,origin_h3,target,request_ts_dt,temperature_2m,relative_humidity_2m,wind_speed_10m,shortwave_radiation,direct_radiation,diffuse_radiation,...,utci_has_heat_stress,utci_has_cold_stress,utci_has_strong_heat_stress,utci_has_strong_cold_stress,qtd_voos_previstos,qtd_empresas_aereas,lista_chegada_real,lista_empresas_aereas,lista_numeros_voo,lista_codigo_linha
0,2025-01-01 04:00:00,8ca901298a137ff,0,2025-01-01 04:14:04,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,2025-01-01 04:00:00,8ca90e91c9669ff,0,2025-01-01 06:50:44,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,2025-01-01 04:00:00,8ca9012998f33ff,0,2025-01-01 06:50:58,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3,2025-01-01 04:00:00,8ca90128a406dff,0,2025-01-01 06:53:45,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
4,2025-01-01 04:00:00,8ca90e82679bbff,0,2025-01-01 06:53:46,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [ ]:
for coluna in df_master.columns:
    print(coluna)

time_window
origin_h3
target
request_ts_dt
temperature_2m
relative_humidity_2m
wind_speed_10m
shortwave_radiation
direct_radiation
diffuse_radiation
direct_normal_irradiance
sunshine_duration
cloud_cover
dew_point_2m
apparent_temperature
precipitation
rain
weather_code
cloud_cover_low
cloud_cover_mid
cloud_cover_high
wind_direction_10m
wind_gusts_10m
surface_pressure
pressure_msl
vapour_pressure_deficit
api_latitude
api_longitude
api_elevation
api_utc_offset_seconds
LAT
LONG
ELEVATION
hour
utci_tdb_c
utci_rh_pct
utci_wind_speed_10m_mps_raw
utci_is_day_estimated
utci_radiative_adjustment_c
utci_tr_c
utci_wind_speed_10m_mps_used
utci_wind_was_clipped
utci_c
utci_discomfort_score_0_100
utci_has_heat_stress
utci_has_cold_stress
utci_has_strong_heat_stress
utci_has_strong_cold_stress
qtd_voos_previstos
qtd_empresas_aereas
lista_chegada_real
lista_empresas_aereas
lista_numeros_voo
lista_codigo_linha


In [ ]:
import pandas as pd

print("1. Criando Features Temporais...")
df_master['hora'] = df_master['time_window'].dt.hour
df_master['mes'] = df_master['time_window'].dt.month
df_master['dia_semana'] = df_master['time_window'].dt.dayofweek
df_master['trimestre'] = df_master['time_window'].dt.quarter

print("2. Criando Features de Lag (Defasagem) no Clima...")
# Ordenar a base agregada de clima por tempo para garantir o shift temporal correto
df_clima_agg = df_clima_agg.sort_values('time_window')

# Selecionar variáveis climáticas chaves para analisar a mudança nas últimas 4 horas
cols_clima_para_lag = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
if 'surface_pressure' in df_clima_agg.columns:
    cols_clima_para_lag.append('surface_pressure')

# Calcular o lag de 1 período (como as janelas são de 4h, lag 1 = 4 horas antes)
for col in cols_clima_para_lag:
    df_clima_agg[f'{col}_lag4h'] = df_clima_agg[col].shift(1)

print("3. Integrando as novas variáveis de lag ao df_master...")
colunas_lags = ['time_window'] + [f'{col}_lag4h' for col in cols_clima_para_lag]

# Realizar o merge usando a mesma janela de tempo
df_master = pd.merge(df_master, df_clima_agg[colunas_lags], on='time_window', how='left')

print("\n--- Amostra das Novas Features ---")
cols_amostra = ['time_window', 'hora', 'mes', 'dia_semana', 'trimestre'] + [f'{col}_lag4h' for col in cols_clima_para_lag]
display(df_master[cols_amostra].head())

1. Criando Features Temporais...
2. Criando Features de Lag (Defasagem) no Clima...
3. Integrando as novas variáveis de lag ao df_master...

--- Amostra das Novas Features ---


,time_window,hora,mes,dia_semana,trimestre,temperature_2m_lag4h,relative_humidity_2m_lag4h,wind_speed_10m_lag4h,surface_pressure_lag4h
0,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
1,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
2,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
3,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
4,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333


In [ ]:
import lightgbm as lgb
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

print("--- Tratamento de Desbalanceamento Severo (Multiclasse) ---")

# 1. Parâmetros Nativos (LightGBM) com Pesos Balanceados
# Calcula os pesos proporcionais para cada classe
classes_unicas = np.unique(df_master['target'])
pesos_balanceados = compute_class_weight(class_weight='balanced', classes=classes_unicas, y=df_master['target'])
class_weight_dict = dict(zip(classes_unicas, pesos_balanceados))

# --- REPONDERAÇÃO MANUAL ---
# Como o undersampling prévio deixou as classes iguais (1:1:1), os pesos automáticos deram 1.0.
# Para penalizar os erros nas classes de evento, podemos ajustar manualmente:
class_weight_dict[0] = 0.5  # Peso menor para 'Não Evento'
class_weight_dict[1] = 1.5  # Peso maior para 'Evento Climático' e Atraso de Voo
class_weight_dict[2] = 1.5  # Mesmo peso

print("\n1. Pesos ajustados para as classes (scale_pos_weight dinâmico/manual):")
for c, w in class_weight_dict.items():
    nome = target_map[c]
    print(f"   {nome}: {w:.4f}")

# Parâmetros base para o LightGBM (Multiclasse)
lgb_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss', # Métrica padrão para multiclasse
    'learning_rate': 0.05,
    'max_depth': 10,
    'random_state': 42
}

print("\n2. Configuração do LightGBM definida:")
for k, v in lgb_params.items():
    print(f"   {k}: {v}")
print("   * Obs: Os pesos das classes serão aplicados na criação do dataset de treino usando o parâmetro 'weight'.")

# 3. Alternativa: Reamostragem (Undersampling)
print("\n3. Configuração de Undersampling (Imbalanced-Learn):")
# Inicializando o undersampler para equilibrar as classes no treino
undersampler = RandomUnderSampler(random_state=42)

print("   * O undersampler pode ser aplicado após a divisão em Treino e Teste (Time-Series Split).")
# Exemplo de uso futuro: X_train_res, y_train_res = undersampler.fit_resample(X_train, y_train)

print("\n✅ Estratégias configuradas! O modelo penalizará muito mais o erro ao errar 'Atrasos de Voo' do que 'Não Eventos'.")

--- Tratamento de Desbalanceamento Severo (Multiclasse) ---

1. Pesos ajustados para as classes (scale_pos_weight dinâmico/manual):
   0 (Não Evento): 0.5000
   1 (Evento Climático): 1.5000
   2 (Atraso de Voo): 1.5000

2. Configuração do LightGBM definida:
   objective: multiclass
   num_class: 3
   metric: multi_logloss
   learning_rate: 0.05
   max_depth: 10
   random_state: 42
   * Obs: Os pesos das classes serão aplicados na criação do dataset de treino usando o parâmetro 'weight'.

3. Configuração de Undersampling (Imbalanced-Learn):
   * O undersampler pode ser aplicado após a divisão em Treino e Teste (Time-Series Split).

✅ Estratégias configuradas! O modelo penalizará muito mais o erro ao errar 'Atrasos de Voo' do que 'Não Eventos'.


In [ ]:
print(f"Período de Treino: {df_train['time_window'].min()} a {df_train['time_window'].max()}")
print(f"Período de Teste:  {df_test['time_window'].min()} a {df_test['time_window'].max()}")

# Colunas não numéricas / não preditivas — NÃO entram no LightGBM
colunas_para_remover = [
    'time_window','origin_h3','target','request_ts_dt',
    'diffuse_radiation','sunshine_duration','dew_point_2m','vapour_pressure_deficit',
    'api_latitude','api_longitude','api_elevation','api_utc_offset_seconds',
    'LAT','LONG','ELEVATION',
    'utci_tdb_c','utci_rh_pct','utci_wind_speed_10m_mps_raw','utci_is_day_estimated',
    'utci_radiative_adjustment_c','utci_tr_c','utci_wind_speed_10m_mps_used',
    'utci_wind_was_clipped','utci_c','utci_discomfort_score_0_100',
    'utci_has_heat_stress','utci_has_cold_stress',
    'utci_has_strong_heat_stress','utci_has_strong_cold_stress',
    'qtd_voos_previstos','qtd_empresas_aereas',
    # colunas de listas — dtype object, incompatível com LightGBM
    'lista_chegada_real','lista_empresas_aereas','lista_numeros_voo','lista_codigo_linha',
]
colunas_para_remover = [col for col in colunas_para_remover if col in df_train.columns]

X_train = df_train.drop(columns=colunas_para_remover)
y_train = df_train['target']

X_test = df_test.drop(columns=colunas_para_remover)
y_test = df_test['target']

print(f"\n✅ Divisão Concluída!")
print(f"Treino (80%): {len(X_train):,} amostras")
print(f"Teste (20%):  {len(X_test):,} amostras")
print(f"Total de features numéricas selecionadas: {X_train.shape[1]}")

In [ ]:
print(f"Período de Treino: {df_train['time_window'].min()} a {df_train['time_window'].max()}")
print(f"Período de Teste:  {df_test['time_window'].min()} a {df_test['time_window'].max()}")

# Definir as colunas que NÃO devem entrar como variáveis preditivas (features)
# As novas colunas de listas são removidas pois o LightGBM só aceita números
colunas_para_remover = [
    'time_window','origin_h3','target','request_ts_dt','diffuse_radiation','sunshine_duration','dew_point_2m','vapour_pressure_deficit','api_latitude','api_longitude','api_elevation','api_utc_offset_seconds','LAT','LONG','ELEVATION','utci_tdb_c','utci_rh_pct','utci_wind_speed_10m_mps_raw','utci_is_day_estimated','utci_radiative_adjustment_c','utci_tr_c','utci_wind_speed_10m_mps_used','utci_wind_was_clipped','utci_c','utci_discomfort_score_0_100','utci_has_heat_stress','utci_has_cold_stress','utci_has_strong_heat_stress','utci_has_strong_cold_stress','qtd_voos_previstos','qtd_empresas_aereas'
]
colunas_para_remover = [col for col in colunas_para_remover if col in df_train.columns]

# Separar Features (X) e Target (y)
X_train = df_train.drop(columns=colunas_para_remover)
y_train = df_train['target']

X_test = df_test.drop(columns=colunas_para_remover)
y_test = df_test['target']

print(f"\n✅ Divisão Concluída!")
print(f"Treino (80%): {len(X_train):,} amostras")
print(f"Teste (20%):  {len(X_test):,} amostras")
print(f"Total de features numéricas selecionadas: {X_train.shape[1]}")


Período de Treino: 2025-01-01 04:00:00 a 2025-10-19 20:00:00
Período de Teste:  2025-10-19 20:00:00 a 2025-12-31 20:00:00

✅ Divisão Concluída!
Treino (80%): 479,659 amostras
Teste (20%):  119,915 amostras
Total de features numéricas selecionadas: 31


In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report, average_precision_score
from sklearn.preprocessing import label_binarize
import numpy as np

print("--- Treinamento e Avaliação do Modelo ---")

# 1. Aplicando os pesos individuais para cada amostra de treino (para forçar o aprendizado dos eventos raros)
sample_weights = y_train.map(class_weight_dict)

# 2. Criando os datasets nativos do LightGBM
dtrain = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
dtest = lgb.Dataset(X_test, label=y_test)

# 3. Treinando o modelo LightGBM
print("Treinando o modelo LightGBM (Isso pode levar alguns segundos)...")
model = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=150,
    valid_sets=[dtrain, dtest],
    callbacks=[lgb.early_stopping(stopping_rounds=15), lgb.log_evaluation(period=50)]
)

# 4. Gerando Previsões no conjunto de Teste (Out-Dez)
# O modelo multiclasse retorna probabilidades para cada classe
y_pred_prob = model.predict(X_test)
# Pegamos a classe com maior probabilidade
y_pred = np.argmax(y_pred_prob, axis=1)

print("\n" + "="*50)
print("📊 RESULTADOS DA AVALIAÇÃO (CONJUNTO DE TESTE)")
print("="*50)

# 5. Precision e Recall (Classification Report)
nomes_classes = ['0 (Não Evento)', '1 (Evento Climático)', '2 (Atraso de Voo)']
print("\n--- Precision & Recall ---")
print(classification_report(y_test, y_pred, target_names=nomes_classes, zero_division=0))

# 6. PR-AUC (Precision-Recall Area Under Curve)
# Binarizando o y_test para calcular a métrica de cada classe separadamente (One-vs-Rest)
print("\n--- PR-AUC (Área sob a Curva Precision-Recall) ---")
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

for i, class_name in enumerate(nomes_classes):
    # Calcula o Average Precision, que é o equivalente prático da PR-AUC
    pr_auc = average_precision_score(y_test_bin[:, i], y_pred_prob[:, i])
    print(f"PR-AUC {class_name}: {pr_auc:.4f}")

print("\n✅ Dica: Avalie principalmente o RECALL e a PR-AUC da classe 2 (Atrasos). Se o Recall estiver alto, significa que estamos detectando grande parte dos atrasos reais!")

--- Treinamento e Avaliação do Modelo ---
Treinando o modelo LightGBM (Isso pode levar alguns segundos)...


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: lista_chegada_real: object, lista_empresas_aereas: object, lista_numeros_voo: object, lista_codigo_linha: object

#PARÂMETROS TREINADOS COM OPTUNA

O modelo do Optuna é a escolha mais defensável porque apresenta:

* maior macro F1;
* maior weighted F1;
* melhor F1 para Não Evento;
* manutenção do F1 de Evento Climático;
* manutenção do F1 de Atraso de Voo;
* menor validation logloss que o terceiro;
* melhor equilíbrio entre as três classes.

In [ ]:
!pip install -q optuna "optuna-integration[lightgbm]"
import optuna
from sklearn.metrics import f1_score
import lightgbm as lgb
import numpy as np

print("--- Otimização de Hiperparâmetros com Optuna ---")

def objective(trial):
    # 1. Definindo o espaço de busca dos hiperparâmetros
    param = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'feature_pre_filter': False, # Necessário para alterar min_data_in_leaf
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'random_state': 42
    }

    # 2. Recria os datasets a cada trial para evitar conflitos de cache do LightGBM
    dtrain_opt = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
    dtest_opt = lgb.Dataset(X_test, label=y_test, reference=dtrain_opt)

    # 3. Treinando o modelo com os parâmetros do trial atual
    # Usando valid_sets para early stopping e evitando overfitting durante a busca
    pruning_callback = optuna.integration.LightGBMPruningCallback(trial, 'multi_logloss')

    model = lgb.train(
        param,
        dtrain_opt,
        num_boost_round=200,
        valid_sets=[dtest_opt],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
    )

    # 4. Avaliando o modelo no conjunto de teste
    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # Como queremos melhorar o F1-score, vamos usar o f1_macro como métrica de otimização
    score = f1_score(y_test, y_pred, average='macro')

    return score

# 5. Configurando e executando o estudo do Optuna
print("Iniciando a busca (isso pode demorar vários minutos)...")
study = optuna.create_study(direction='maximize') # Queremos maximizar o F1-score
study.optimize(objective, n_trials=30)

print("\n✅ Otimização Concluída!")
print("Melhor F1-score encontrado:", study.best_value)
print("Melhores hiperparâmetros:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"    {key}: {value}")

# 6. Adicionando parâmetros fixos aos melhores encontrados
best_params['objective'] = 'multiclass'
best_params['num_class'] = 3
best_params['metric'] = 'multi_logloss'
best_params['random_state'] = 42
best_params['feature_pre_filter'] = False

# Agora você pode retreinar o modelo final com best_params
print("\n--- Os hiperparâmetros otimizados estão armazenados na variável 'best_params' ---")

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

print("--- Curvas Precision-Recall (Modelo Final) ---")

fig, ax = plt.subplots(figsize=(10, 7))

for i, class_name in enumerate(nomes_classes):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_pred_prob_final[:, i])
    pr_auc = average_precision_score(y_test_bin[:, i], y_pred_prob_final[:, i])
    ax.plot(recall, precision, lw=2, label=f'{class_name} (AUC = {pr_auc:.4f})')

ax.set_title('Curvas Precision-Recall Multiclasse', fontsize=14)
ax.set_xlabel('Recall (Taxa de Verdadeiros Positivos)', fontsize=12)
ax.set_ylabel('Precision (Valor Preditivo Positivo)', fontsize=12)
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

print("--- Curvas Precision-Recall (Modelo Recalibrado) ---")

fig, ax = plt.subplots(figsize=(10, 7))

# Loop para plotar a curva de cada classe (One-vs-Rest)
for i, class_name in enumerate(nomes_classes):
    # Calcula a precisão e recall para vários limiares (thresholds)
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_pred_prob_recal[:, i])

    # Calcula a área sob a curva (já calculada antes, mas repetimos para a legenda do gráfico)
    pr_auc = average_precision_score(y_test_bin[:, i], y_pred_prob_recal[:, i])

    # Plota a curva
    ax.plot(recall, precision, lw=2, label=f'{class_name} (AUC = {pr_auc:.4f})')

# Configurações de exibição do gráfico
ax.set_title('Curvas Precision-Recall Multiclasse - Modelo Recalibrado', fontsize=14)
ax.set_xlabel('Recall (Taxa de Verdadeiros Positivos)', fontsize=12)
ax.set_ylabel('Precision (Valor Preditivo Positivo)', fontsize=12)
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### 💾 Salvando o Modelo para Uso Futuro
Para não precisar retreinar o modelo toda vez, salvamos ele em um arquivo. Também é uma excelente prática salvar a **lista exata de features (colunas)** para garantir que no futuro não falte nada na sua base.

In [ ]:
import joblib
import lightgbm as lgb

print("--- 1. Salvando os artefatos do modelo ---")

# Salvando o modelo LightGBM nativo
nome_arquivo_modelo = 'modelo_lgb_atrasos_recalibrado_v1.txt'
final_model.save_model(nome_arquivo_modelo)
print(f"✅ Modelo salvo com sucesso: {nome_arquivo_modelo}")

# Salvando a lista de colunas que o modelo espera
features_esperadas = X_train.columns.tolist()
joblib.dump(features_esperadas, 'features_esperadas_v1.pkl')
print("✅ Lista de features salva com sucesso: features_esperadas_v1.pkl")

# Opcional: Se quiser salvar no Google Drive para não perder quando o Colab fechar:
# !cp modelo_lgb_atrasos_recalibrado_v1.txt /content/drive/MyDrive/DOUTORADO/
# !cp features_esperadas_v1.pkl /content/drive/MyDrive/DOUTORADO/

### 🚀 Simulando Previsão em Tempo Real (Produção)
Como carregar o modelo em outro arquivo Python ou Notebook e prever a próxima janela de 4h.

In [ ]:
import numpy as np

print("--- 2. Carregando e Prevendo Dados Futuros ---")

# A) Carregando o modelo e as features
modelo_carregado = lgb.Booster(model_file='modelo_lgb_atrasos_recalibrado_v1.txt')
colunas_modelo = joblib.load('features_esperadas_v1.pkl')

# B) Simulando a chegada de um "Dado Futuro"
# Na vida real, aqui você rodaria seu pipeline de engenharia de dados (SQL, Pandas)
# para pegar a previsão do tempo das próximas 4h, a quantidade de voos, os lags, etc.
# Para simular, vou pegar aleatoriamente a linha 500 do nosso conjunto de teste atual.
dados_futuros = X_test.iloc[[10]].copy()

# Garante que os dados futuros tenham exatamente as mesmas colunas, na mesma ordem do treino
dados_futuros = dados_futuros[colunas_modelo]

# C) Fazendo a previsão
probabilidades_futuras = modelo_carregado.predict(dados_futuros)
classe_prevista = np.argmax(probabilidades_futuras, axis=1)[0]

print("\n🔮 PREVISÃO PARA A PRÓXIMA JANELA:")
print(f"   -> Probabilidade de Não Evento (0):       {probabilidades_futuras[0][0]:.1%}")
print(f"   -> Probabilidade de Evento Climático (1): {probabilidades_futuras[0][1]:.1%}")
print(f"   -> Probabilidade de Atraso de Voo (2):    {probabilidades_futuras[0][2]:.1%}")
print("-"*40)
print(f"🚨 DECISÃO DO ALGORITMO: {target_map[classe_prevista]}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("--- Importância das Variáveis (Feature Importance) ---")

# Extraindo a importância (baseada no ganho de informação)
importances = final_model_recalibrated.feature_importance(importance_type='gain')
feature_names = final_model_recalibrated.feature_name()

# Criando um DataFrame para facilitar a visualização
df_importances = pd.DataFrame({'feature': feature_names, 'importance': importances})
df_importances = df_importances.sort_values(by='importance', ascending=False)

# Plotando as 20 variáveis mais importantes
plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', hue='feature', data=df_importances.head(20), palette='viridis', legend=False)
plt.title('Top 20 Variáveis Mais Importantes (Ganho de Informação)', fontsize=14)
plt.xlabel('Importância (Gain)', fontsize=12)
plt.ylabel('Variável', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

print("--- 2. Carregando e Prevendo Dados Futuros ---")

# A) Carregando o modelo e as features
modelo_carregado = lgb.Booster(model_file='modelo_lgb_atrasos_recalibrado_v1.txt')
colunas_modelo = joblib.load('features_esperadas_v1.pkl')

# B) Simulando a chegada de um "Dado Futuro"
# Na vida real, aqui você rodaria seu pipeline de engenharia de dados (SQL, Pandas)
# para pegar a previsão do tempo das próximas 4h, a quantidade de voos, os lags, etc.
# Para simular, vou pegar aleatoriamente a linha 500 do nosso conjunto de teste atual.
dados_futuros = X_test.iloc[[10]].copy()

# Garante que os dados futuros tenham exatamente as mesmas colunas, na mesma ordem do treino
dados_futuros = dados_futuros[colunas_modelo]

# C) Fazendo a previsão
probabilidades_futuras = modelo_carregado.predict(dados_futuros)
classe_prevista = np.argmax(probabilidades_futuras, axis=1)[0]

print("\n🔮 PREVISÃO PARA A PRÓXIMA JANELA:")
print(f"   -> Probabilidade de Não Evento (0):       {probabilidades_futuras[0][0]:.1%}")
print(f"   -> Probabilidade de Evento Climático (1): {probabilidades_futuras[0][1]:.1%}")
print(f"   -> Probabilidade de Atraso de Voo (2):    {probabilidades_futuras[0][2]:.1%}")
print("-"*40)
print(f"🚨 DECISÃO DO ALGORITMO: {target_map[classe_prevista]}")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("--- Análise de Erros: Matriz de Confusão ---")

cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nomes_classes, yticklabels=nomes_classes)
plt.title('Matriz de Confusão', fontsize=14)
plt.ylabel('Classe Real', fontsize=12)
plt.xlabel('Classe Prevista pelo Modelo', fontsize=12)
plt.show()

print("\n🔎 DETALHAMENTO DOS ERROS DA CLASSE 2 (ATRASO DE VOO):")
falsos_positivos_atraso = cm[0, 2] + cm[1, 2]
falsos_negativos_atraso = cm[2, 0] + cm[2, 1]
print(f"- Falsos Positivos (Alarme Falso de Atraso): {falsos_positivos_atraso:,}")
print(f"- Falsos Negativos (Atraso Perdido):         {falsos_negativos_atraso:,}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("--- Análise de Erros: Matriz de Confusão ---")

# Calculando a matriz de confusão com as previsões otimizadas pelo limiar (best_y_pred)
cm = confusion_matrix(y_test, best_y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=nomes_classes, yticklabels=nomes_classes)
plt.title('Matriz de Confusão (Com Limiares Otimizados)', fontsize=14)
plt.ylabel('Classe Real', fontsize=12)
plt.xlabel('Classe Prevista pelo Modelo', fontsize=12)
plt.show()

print("\n🔎 DETALHAMENTO DOS ERROS DA CLASSE 2 (ATRASO DE VOO):")
falsos_positivos_atraso = cm[0, 2] + cm[1, 2]
falsos_negativos_atraso = cm[2, 0] + cm[2, 1]
print(f"- Falsos Positivos (Alarme Falso de Atraso): {falsos_positivos_atraso:,} vezes o modelo previu atraso, mas não ocorreu.")
print(f"- Falsos Negativos (Atraso Perdido):         {falsos_negativos_atraso:,} vezes ocorreu atraso, mas o modelo não previu.")


### 🛠️ Frente 4: Calibração de Probabilidades
Vamos verificar quão calibradas estão as probabilidades do nosso modelo e aplicar a regressão isotônica para ajustá-las.

In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from lightgbm import LGBMClassifier

print("--- Calibração de Probabilidades ---")

# Para usar o CalibratedClassifierCV nativamente, é mais fácil usar a API do Scikit-Learn do LightGBM.
# Como já temos as probabilidades finais (y_pred_prob_final), vamos visualizar a curva de calibração atual da classe 2 (Atrasos)

prob_true, prob_pred = calibration_curve(y_test == 2, y_pred_prob_final[:, 2], n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', label='LightGBM (Atual)')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Calibração Perfeita (Ideal)')
plt.title('Curva de Calibração (Reliability Diagram) - Classe: Atraso de Voo', fontsize=14)
plt.xlabel('Probabilidade Prevista Média', fontsize=12)
plt.ylabel('Fração Real de Positivos', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print("\n💡 INTERPRETAÇÃO:")
print("Se a linha azul estiver muito distante da linha pontilhada cinza, significa que o modelo está muito 'confiante' ou muito 'inseguro'.\nA calibração corrigirá isso para que uma previsão de 80% signifique realmente 80% de chance.")


### 🚀 Engenharia de Features Avançada
Criando derivadas de clima (tendências) e detalhando a malha aérea a partir das listas agregadas.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report, average_precision_score

print("--- Re-treinando o Modelo com as Novas Features ---")

colunas_para_remover_adv = [
    'time_window','origin_h3','target','request_ts_dt','diffuse_radiation','sunshine_duration',
    'dew_point_2m','vapour_pressure_deficit','api_latitude','api_longitude','api_elevation',
    'api_utc_offset_seconds','LAT','LONG','ELEVATION','utci_tdb_c','utci_rh_pct',
    'utci_wind_speed_10m_mps_raw','utci_is_day_estimated','utci_radiative_adjustment_c',
    'utci_tr_c','utci_wind_speed_10m_mps_used','utci_wind_was_clipped','utci_c',
    'utci_discomfort_score_0_100','utci_has_heat_stress','utci_has_cold_stress',
    'utci_has_strong_heat_stress','utci_has_strong_cold_stress','qtd_voos_previstos','qtd_empresas_aereas',
    'lista_chegada_real','lista_empresas_aereas','lista_numeros_voo','lista_codigo_linha',
]

df_train_adv = df_master.iloc[:indice_corte].copy()
df_test_adv  = df_master.iloc[indice_corte:].copy()

colunas_para_remover_adv = [col for col in colunas_para_remover_adv if col in df_train_adv.columns]

X_train_adv = df_train_adv.drop(columns=colunas_para_remover_adv)
y_train_adv = df_train_adv['target']
X_test_adv  = df_test_adv.drop(columns=colunas_para_remover_adv)
y_test_adv  = df_test_adv['target']

# Pesos usando y_train_adv (não y_train do split anterior)
sample_weights_adv = y_train_adv.map(class_weight_dict)

dtrain_adv = lgb.Dataset(X_train_adv, label=y_train_adv, weight=sample_weights_adv,
                         params={'feature_pre_filter': False})
dtest_adv  = lgb.Dataset(X_test_adv,  label=y_test_adv,  reference=dtrain_adv,
                         params={'feature_pre_filter': False})

print("Iniciando treinamento...")
model_adv = lgb.train(
    best_params,
    dtrain_adv,
    num_boost_round=200,
    valid_sets=[dtrain_adv, dtest_adv],
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=50)],
)

y_pred_prob_adv = model_adv.predict(X_test_adv)
y_pred_adv      = np.argmax(y_pred_prob_adv, axis=1)

print("\n" + "="*50)
print("📊 RESULTADOS COM ENGENHARIA DE FEATURES AVANÇADA")
print("="*50)
print("\n--- Precision & Recall ---")
print(classification_report(y_test_adv, y_pred_adv, target_names=nomes_classes, zero_division=0))

print("\n--- PR-AUC ---")
for i, class_name in enumerate(nomes_classes):
    pr_auc = average_precision_score(y_test_bin[:, i], y_pred_prob_adv[:, i])
    print(f"PR-AUC {class_name}: {pr_auc:.4f}")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report, average_precision_score

print("--- Re-treinando o Modelo com as Novas Features ---")

# Atualiza a lista de colunas para remover (precisamos remover as listas cruas para não dar erro no LightGBM)
colunas_para_remover_adv = [
    'time_window','origin_h3','target','request_ts_dt','diffuse_radiation','sunshine_duration',
    'dew_point_2m','vapour_pressure_deficit','api_latitude','api_longitude','api_elevation',
    'api_utc_offset_seconds','LAT','LONG','ELEVATION','utci_tdb_c','utci_rh_pct',
    'utci_wind_speed_10m_mps_raw','utci_is_day_estimated','utci_radiative_adjustment_c',
    'utci_tr_c','utci_wind_speed_10m_mps_used','utci_wind_was_clipped','utci_c',
    'utci_discomfort_score_0_100','utci_has_heat_stress','utci_has_cold_stress',
    'utci_has_strong_heat_stress','utci_has_strong_cold_stress','qtd_voos_previstos','qtd_empresas_aereas',
    'lista_chegada_real','lista_empresas_aereas','lista_numeros_voo','lista_codigo_linha'
]

# Refazendo o split (usando o mesmo índice de corte temporal definido anteriormente)
df_train_adv = df_master.iloc[:indice_corte].copy()
df_test_adv = df_master.iloc[indice_corte:].copy()

colunas_para_remover_adv = [col for col in colunas_para_remover_adv if col in df_train_adv.columns]

X_train_adv = df_train_adv.drop(columns=colunas_para_remover_adv)
y_train_adv = df_train_adv['target']
X_test_adv = df_test_adv.drop(columns=colunas_para_remover_adv)
y_test_adv = df_test_adv['target']

# Treinando o modelo
dtrain_adv = lgb.Dataset(X_train_adv, label=y_train_adv, weight=sample_weights, params={'feature_pre_filter': False})
dtest_adv = lgb.Dataset(X_test_adv, label=y_test_adv, reference=dtrain_adv, params={'feature_pre_filter': False})

print("Iniciando treinamento...")
model_adv = lgb.train(
    best_params, # Reutilizamos os melhores parâmetros encontrados pelo Optuna
    dtrain_adv,
    num_boost_round=200,
    valid_sets=[dtrain_adv, dtest_adv],
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=50)]
)

# Previsões
y_pred_prob_adv = model_adv.predict(X_test_adv)
y_pred_adv = np.argmax(y_pred_prob_adv, axis=1)

print("\n" + "="*50)
print("📊 RESULTADOS COM ENGENHARIA DE FEATURES AVANÇADA")
print("="*50)
print("\n--- Precision & Recall ---")
print(classification_report(y_test_adv, y_pred_adv, target_names=nomes_classes, zero_division=0))

print("\n--- PR-AUC ---")
for i, class_name in enumerate(nomes_classes):
    pr_auc = average_precision_score(y_test_bin[:, i], y_pred_prob_adv[:, i])
    print(f"PR-AUC {class_name}: {pr_auc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Nova Importância das Variáveis (Com Features Avançadas) ---")

# Extraindo a importância do novo modelo
importances_adv = model_adv.feature_importance(importance_type='gain')
feature_names_adv = model_adv.feature_name()

df_importances_adv = pd.DataFrame({'feature': feature_names_adv, 'importance': importances_adv})
df_importances_adv = df_importances_adv.sort_values(by='importance', ascending=False)

# Plotando as 20 variáveis mais importantes
plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', hue='feature', data=df_importances_adv.head(20), palette='viridis', legend=False)
plt.title('Top 20 Variáveis Mais Importantes (Com Features Avançadas)', fontsize=14)
plt.xlabel('Importância (Gain)', fontsize=12)
plt.ylabel('Variável', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

print("--- Realizando 5 Previsões e Calculando a Taxa de Acerto ---")

# 1. Selecionando uma amostra de 300 casos do conjunto de teste avançado
amostra_X = X_test_adv.sample(n=30000, random_state=42)
amostra_y = y_test_adv.loc[amostra_X.index]

# 2. Fazendo as previsões com o modelo de features avançadas
probabilidades = model_adv.predict(amostra_X)
previsoes = np.argmax(probabilidades, axis=1)

# 3. Calculando a taxa de acerto (Acurácia)
taxa_acerto = accuracy_score(amostra_y, previsoes)
acertos = np.sum(amostra_y.values == previsoes)

print(f"Total de previsões realizadas: {len(amostra_X)}")
print(f"Total de acertos: {acertos}")
print(f"Taxa de Acerto (Acurácia) na amostra: {taxa_acerto:.2%}\n")

# 4. Criando um DataFrame para comparar visualmente os resultados
comparativo = pd.DataFrame({
    'Classe_Real': amostra_y.values,
    'Classe_Prevista': previsoes
})

# Mapeando os nomes para facilitar a leitura
comparativo['Real_Descricao'] = comparativo['Classe_Real'].map(target_map)
comparativo['Previsto_Descricao'] = comparativo['Classe_Prevista'].map(target_map)
comparativo['Acertou?'] = comparativo['Classe_Real'] == comparativo['Classe_Prevista']

print("Amostra dos 15 primeiros resultados:")
display(comparativo.head(15))